most similar objects with cosine similarity

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import torch
from PIL import Image

EMBED_PATH = Path('multimodal_embeddings_gemini_2.pt')
IMAGES_DIR = Path('images')
QUERY_OBJECT_ID = '4002552445'
TOP_K = 3

In [ ]:
db = torch.load(EMBED_PATH, map_location='cpu')
embeddings = db['embeddings'].float()
rows = db['rows']

# normalize vectors so dot product -> cosine similarity
embeddings = embeddings / embeddings.norm(dim=1, keepdim=True)

object_ids = [row['object_id'] for row in rows]
if QUERY_OBJECT_ID not in object_ids:
    raise ValueError(f'Unknown object_id: {QUERY_OBJECT_ID}')

query_idx = object_ids.index(QUERY_OBJECT_ID)

print('embeddings shape:', tuple(embeddings.shape))
print('objects:', len(rows))
print('query object_id:', QUERY_OBJECT_ID)

In [ ]:
query_row = rows[query_idx]
query_images = sorted((IMAGES_DIR / QUERY_OBJECT_ID).glob('*.jpg'))

print('QUERY OBJECT')
print('object_id:', QUERY_OBJECT_ID)
print('text_prompt:\n', query_row['text_prompt'])
print('n_images:', len(query_images))

for p in query_images:
    img = Image.open(p).convert('RGB')
    plt.figure(figsize=(4, 4))
    plt.imshow(img)
    plt.title(p.name, fontsize=9)
    plt.axis('off')
    plt.show()

In [ ]:
q = embeddings[query_idx]
sims = embeddings @ q
sims[query_idx] = -1  # remove the query itself

top_scores, top_idxs = torch.topk(sims, k=TOP_K)

print('TOP 3 MOST SIMILAR OBJECTS')
for i in range(TOP_K):
    idx = top_idxs[i].item()
    score = top_scores[i].item()
    print(f"#{i+1} | object_id={rows[idx]['object_id']} | cosine={score:.4f}")

In [ ]:
for i in range(TOP_K):
    idx = top_idxs[i].item()
    score = top_scores[i].item()
    row = rows[idx]
    object_id = row['object_id']

    print('=' * 80)
    print(f'SIMILAR OBJECT #{i+1}')
    print('object_id:', object_id)
    print(f'cosine similarity: {score:.4f}')
    print('text_prompt:\n', row['text_prompt'])

    image_paths = sorted((IMAGES_DIR / object_id).glob('*.jpg'))
    print('n_images:', len(image_paths))

    for p in image_paths:
        img = Image.open(p).convert('RGB')
        plt.figure(figsize=(4, 4))
        plt.imshow(img)
        plt.title(p.name, fontsize=9)
        plt.axis('off')
        plt.show()